# AIFM Hedge Fund — Risk Notebook

AIFM Risk limits are defined in the fund's offering document and monitored against
internal thresholds. No regulatory VaR limit applies (unlike UCITS).

Key regulatory obligations under AIFMD:
- **Leverage**: gross and commitment method (Annex IV)
- **Stress testing**: market, liquidity, and counterparty scenarios (Annex VI)
- **Liquidity risk**: portfolio liquidity profile and redemption stress
- **Annex IV reporting**: quarterly to CSSF. AIFMD II (Directive 2024/927/EU)
  expanded requirements, adding granular data on liquidity management tools,
  loan origination, and delegation arrangements.

#### In this notebook

AIFM Hedge Fund, Strategy: Long/short equity, bonds, FX, options.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.setup_db import run as setup_db
setup_db()

from src.plot_style import (
    C, BUCKET_COLORS, FUND_COLORS, ACCENT, ACCENT2, ACCENT3, WARNING,
    apply_ax_style, section_title, fig_header, fig_footer,
    callout_box, threshold_vline, threshold_hline, breach_fill,
    pct_color, rag_color, util_color, liq_color, table_cell,sup_title,
)
from src.database import get_engine, query_positions, query_nav_history
from src.enrichment import get_risk_ready_df
from src.mock_bloomberg import MockBloomberg as Bloomberg
from src.leverage_config import INSTRUMENT_SOURCE
from src.risk_utils import (
    var_historical, var_parametric, var_scale, var_montecarlo,
    es_historical, es_scale,
    exception_report, full_backtest_report,
    stress_equity, stress_rates, stress_credit,
    stress_fx, stress_combined, stress_historical,
    days_to_liquidate, liquidity_buckets, redemption_stress, investor_concentration,
    liquidity_adjusted_var,
)
from src.esg_utils import build_esg_df, esg_portfolio_summary, ESG_FIELDS, ESG_THRESHOLD
from src.nb_utils import save_fig, save_table, styled_table


FUND_ID    = 'AIFM_HedgeFund'
VALUATION_DATE      = '2026-05-13'
ENGINE     = get_engine()
BBG        = Bloomberg()

CONFIDENCE = 0.99
HORIZON    = 20


---

## 1. Load and Validate Positions

Positions are queried from SQLite, which is loaded daily from the fund administrator
Excel export. The flow is:

Fund admin Excel → load_positions() → SQLite → query_positions() → notebook

`get_risk_ready_df` runs the enrichment pipeline on the raw positions:
- liquid instruments (equities, bonds, ETFs): sensitivities fetched from Bloomberg
  (beta, modified duration, convexity, spread duration)
- illiquid instruments (loans, direct properties): fund admin data already embedded
  in the position file (rating, maturity, LTV, rental yield) used directly,
  no Bloomberg ticker available or needed

The output is a single enriched DataFrame per fund per date, ready for VaR,
stress testing, and liquidity analysis.

In [ ]:
positions = query_positions(ENGINE, FUND_ID, VALUATION_DATE)
risk_df   = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)
NAV       = risk_df['market_value_eur'].sum()

print(f"Fund           : {FUND_ID}")
print(f"Valuation date : {VALUATION_DATE}")
print(f"Positions      : {len(positions)}")
print(f"NAV (EUR)      : {NAV:,.0f}")
print(f"Asset classes  : {sorted(positions['asset_class'].unique())}")
print(f"Long exposure  : {risk_df[risk_df['market_value_eur'] > 0]['market_value_eur'].sum():,.0f}")
print(f"Short exposure : {risk_df[risk_df['market_value_eur'] < 0]['market_value_eur'].sum():,.0f}")

In [ ]:
# Asset class breakdown
breakdown = risk_df.groupby('asset_class').agg(
    market_value_eur=('market_value_eur', 'sum'),
    n_positions=('isin', 'count'),
).sort_values('market_value_eur', ascending=False)

breakdown['weight_pct'] = breakdown['market_value_eur'] / NAV * 100

print(f"{'Asset Class':<20} {'MV (EUR)':>15} {'Weight':>8} {'# Pos':>6}")
print('-' * 52)
for ac, row in breakdown.iterrows():
    print(f"{ac:<20} {row['market_value_eur']:>15,.0f} {row['weight_pct']:>7.1f}% {row['n_positions']:>6}")
print('-' * 52)
print(f"{'NAV':<20} {NAV:>15,.0f} {'100.0%':>8}")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
colors = [ACCENT2 if v < 0 else ACCENT for v in breakdown['weight_pct']]
bars = ax.barh(breakdown.index, breakdown['weight_pct'],
               color=colors, height=0.45, alpha=0.85)
ax.axvline(0, color=C['dim'], lw=0.8)
ax.set_xlabel('Weight (% NAV)', fontsize=9)
section_title(ax, 'Asset Class Breakdown')
ax.grid(True, axis='x', alpha=0.15, linestyle='--')
for bar, val in zip(bars, breakdown['weight_pct']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=7)
plt.tight_layout()
save_fig(fig, FUND_ID, "01. Asset Breakdown HF", dpi=150)
plt.show()

---

## 2. VaR and Expected Shortfall

VaR and ES are computed using Historical and Montecarlo simulation on daily NAV returns.
Under AIFMD there is no regulatory VaR limit, but VaR is monitored against
internal limits defined in the RMP and reported in Annex IV.

- Confidence: 99%
- Horizon: 1-day and 20-day (square root of time scaling)
- Method: historical simulation, 250-day rolling window

> **Monte Carlo VaR**: in production these figures come from a third-party risk
> system (Bloomberg PORT, Aladdin, Axioma) and are consumed directly by the risk
> manager. To preserve the workflow without a live risk system connection, we
> approximate using a multi-factor simulation with a hardcoded covariance matrix
> and first-order sensitivities.

In [ ]:
nav_history = query_nav_history(ENGINE, FUND_ID)
nav_history['date'] = pd.to_datetime(nav_history['date'])
pnl = nav_history['pnl_pct'].dropna().values

# historical simulation
var_1d  = var_historical(pnl[-250:], confidence=CONFIDENCE)
var_20d = var_scale(var_1d, horizon=HORIZON)
es_1d   = es_historical(pnl[-250:], confidence=CONFIDENCE)
es_20d  = es_scale(es_1d, horizon=HORIZON)

# monte carlo
mc      = var_montecarlo(risk_df, n_sims=10_000,
                         confidence=CONFIDENCE, horizon=1)
mc_20d  = var_scale(mc['var'], horizon=HORIZON)

print(f"{'Metric':<25} {'1d':>10} {'20d':>10}")
print(f"{'':25} {'(% NAV)':>10} {'(% NAV)':>10}")
print('-' * 46)
print(f"{'VaR Historical':<25} {var_1d*100:>9.2f}% {var_20d*100:>9.2f}%")
print(f"{'VaR Monte Carlo':<25} {mc['var']/NAV*100:>9.2f}% {mc_20d/NAV*100:>9.2f}%")
print(f"{'ES Historical':<25} {es_1d*100:>9.2f}% {es_20d*100:>9.2f}%")
print(f"{'ES Monte Carlo':<25} {mc['es']/NAV*100:>9.2f}% {var_scale(mc['es'],HORIZON)/NAV*100:>9.2f}%")
print('-' * 46)
print(f"{'VaR Hist (EUR)':<25} {var_1d*NAV:>10,.0f} {var_20d*NAV:>10,.0f}")
print(f"{'VaR MC (EUR)':<25} {mc['var']:>10,.0f} {mc_20d:>10,.0f}")

### 2.2 Liquidity-Adjusted VaR

Standard VaR assumes positions can be liquidated instantly at market price.
LVaR extends this by adding the estimated cost of unwinding positions under
stressed market conditions:

$$\text{LVaR} = \text{VaR} + \text{Liquidity Cost}$$

$$\text{Liquidity Cost}_i = \frac{1}{2} \times \text{spread}_i \times \text{stress multiplier}_i \times |MV_i|$$

LVaR is not a regulatory requirement. It originates from banking regulation
(Basel III internal models) and academic literature (Amihud & Mendelson, BIS
working papers) and is used internally by risk managers to capture the
liquidity dimension of market risk.

Spreads and stress multipliers are illustrative values adopted in this
notebook. In practice these are internal RMP parameters calibrated by the
fund manager and reviewed periodically.

| Asset Class      | Normal Spread | Stress Multiplier | Stressed Spread       |
|------------------|---------------|-------------------|-----------------------|
| Large cap equity | 5bps          | 3x                | 15bps                 |
| Small cap equity | 20bps         | 5x                | 100bps                |
| IG bonds         | 10bps         | 5x                | 50bps                 |
| HY bonds         | 50bps         | 10x               | 500bps                |
| Senior loans     | 100bps        | 20x               | 2000bps               |
| Listed REITs     | 15bps         | 5x                | 75bps                 |
| Direct property  | N/A           | N/A               | 5-8% transaction cost |

Mandatory liquidity risk management (buckets, redemption stress, LMTs) is
covered in Section 5.

In [ ]:
# MRS-33: Liquidity-adjusted VaR
lvar_result = liquidity_adjusted_var(var_1d, risk_df)

print(f"VaR (1d 99%)        : {lvar_result['var']*100:.2f}%   EUR {lvar_result['var']*NAV:,.0f}")
print(f"Liquidity cost      : {lvar_result['liquidity_cost']*100:.2f}%   EUR {lvar_result['liquidity_cost']*NAV:,.0f}")
print(f"LVaR (1d 99%)       : {lvar_result['lvar']*100:.2f}%   EUR {lvar_result['lvar']*NAV:,.0f}")
print(f"LVaR increase       : +{lvar_result['lvar_pct_increase']:.1f}%")
print()
print(lvar_result['by_asset_class'].to_string())

---

## 3. VaR Backtest (Kupiec + Christoffersen + ESMA)

VaR backtesting compares predicted VaR against realised daily P&L.
Two statistical tests are applied:

- **Kupiec POF**: tests whether the breach rate equals the expected rate (1% for 99% VaR)
- **Christoffersen**: tests whether breaches are independently distributed over time

> **Note**: Kupiec and Christoffersen tests are industry best practice and not explicitly
> required by ESMA or CSSF. The regulatory requirement is the 250-day breach count
> and zone classification only. Both tests should be documented in the RMP. For both
> models p > 0.05: fail to reject the null, model is statistically acceptable.

> **Private debt note**: NAV returns reflect mark-to-model valuations from the fund
> administrator, which smooth daily volatility. Breach counts will likely be lower
> than for a liquid fund. Results should be interpreted with this in mind.

ESMA (CESR/10-788) requires backtesting 1-day 99% VaR against realised daily P&L
over a 250 trading day rolling window.

Zone classification (Basel traffic light, adopted by ESMA):
- **Green** (0-4 breaches): model acceptable
- **Amber** (5-9 breaches): explanation required, possible model review
- **Red** (10+ breaches): model must be revised, CSSF notification required

#### 3.1. Internal (maximum history)

In [ ]:
# MRS-27: VaR backtest
nav_history = query_nav_history(ENGINE, FUND_ID)
nav_history['date'] = pd.to_datetime(nav_history['date'])

pnl   = nav_history['pnl_pct'].dropna().values
dates = nav_history['date'].iloc[1:].reset_index(drop=True)

window        = 250
var_hist      = pd.Series([
    var_historical(pnl[max(0, i-window):i], confidence=0.99)
    for i in range(window, len(pnl))
])
var_param     = pd.Series([
    var_parametric(mu=0, sigma=pnl[max(0, i-window):i].std(),
                   confidence=0.99, dist='t')
    for i in range(window, len(pnl))
])

pnl_aligned   = pnl[window:]
dates_aligned = dates.iloc[window:].reset_index(drop=True)
print(f"Backtest observations : {len(pnl_aligned)}")

In [ ]:
report = full_backtest_report(
    pnl_series=pd.Series(pnl_aligned),
    var_dict={'Historical': var_hist, 'Parametric': var_param},
    dates=dates_aligned
)
report[['n_obs', 'n_breaches', 'breach_rate', 'expected',
        'kupiec_p', 'christoffersen_p', 'result']]

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates_aligned, pnl_aligned * 100,
        color=C['muted'], lw=0.8, label='Daily P&L', alpha=0.7)
ax.plot(dates_aligned, -var_hist * 100,
        color=ACCENT, lw=1.2, label='99% VaR (historical)')
ax.plot(dates_aligned, -var_param * 100,
        color=ACCENT3, lw=1.2, label='99% VaR (parametric)', linestyle='--')

breaches = pnl_aligned < -var_hist.values
ax.scatter(dates_aligned[breaches], pnl_aligned[breaches] * 100,
           color=ACCENT2, s=25, zorder=5, label=f'Breaches ({breaches.sum()})')

ax.set_ylabel('Daily P&L / VaR (%)', fontsize=9)
section_title(ax, f'VaR Backtest — {FUND_ID}', fontsize=18)
ax.legend(fontsize=10)
plt.tight_layout()
save_fig(fig, FUND_ID, "02. VAR backtest HF - full history")
plt.show()

#### 3.2. ESMA (250 trading days)

In [ ]:
esma_250  = exception_report(pd.Series(pnl_aligned[-250:]),
                              var_hist.iloc[-250:], confidence=0.99)
n_250     = len(esma_250)
breach_250 = n_250 / 250
zone_250  = 'Green' if n_250 <= 4 else 'Amber' if n_250 <= 9 else 'Red'

print(f"ESMA regulatory window — last 250 trading days")
print(f"Breaches    : {n_250}")
print(f"Breach rate : {breach_250*100:.2f}% (expected 1.0%)")
print(f"ESMA zone   : {zone_250}")
print()


In [ ]:
dates_250 = dates_aligned.iloc[-250:].reset_index(drop=True)
pnl_250   = pnl_aligned[-250:]
var_250   = var_hist.iloc[-250:].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates_250, pnl_250 * 100,
        color=C['muted'], lw=1.2, label='Daily P&L', alpha=0.7)
ax.plot(dates_250, -var_250 * 100,
        color=ACCENT, lw=1.2, label='99% VaR (historical)')

breaches_250 = pnl_250 < -var_250.values
ax.scatter(dates_250[breaches_250], pnl_250[breaches_250] * 100,
           color=ACCENT2, s=30, zorder=5, label=f'Breaches ({n_250})')

zone_color = {'Green': C['green'], 'Amber': C['amber'], 'Red': C['red']}

section_title(ax, f'ESMA Exception Report — Last 250 Days — Zone: {zone_250}', fontsize=18)

ax.set_ylabel('Daily P&L / VaR (%)', fontsize=9)
ax.legend(fontsize=10)
plt.tight_layout()
save_fig(fig, FUND_ID, "03. VAR backtest HF - last 250d")
plt.show()

---

## 4. Leverage (Annex IV)

AIFMD requires leverage to be reported using two methods:

- **Gross method**: sum of absolute exposures divided by NAV. No netting allowed.
  Derivatives converted to equivalent underlying exposure.
- **Commitment method**: hedging and netting arrangements are recognised.
  Offsetting positions in the same underlying reduce exposure.

Limits are set in the fund's offering document and reported quarterly to the CSSF
in Annex IV. AIFMD II (Directive 2024/927/EU) expanded reporting requirements,
including the breakdown by:
* asset class
* instrument type
* source: financial borrowing, synthetic leverage through derivatives, and repo/reverse repo.

The expanded disclosure makes it easier for regulators to identify funds building leverage through
derivatives rather than borrowing.

In [ ]:
# MRS-23: Leverage computation - Gross and Commitment method

# ----------------------------------------------------------------
# Gross method: sum of absolute exposures / NAV
# ----------------------------------------------------------------
risk_df['abs_exposure'] = risk_df['market_value_eur'].abs()

deriv_rows     = risk_df[risk_df['asset_class'] == 'Derivative'].copy()
deriv_notional = 0.0
for _, row in deriv_rows.iterrows():
    ticker        = 'SPXW 260619P05500 Index'
    bbg_data      = BBG.bdp(ticker, ['DELTA', 'OPT_UNDL_PX', 'CONTRACT_SIZE'])
    delta         = abs(bbg_data.loc[ticker, 'DELTA'])
    undl_px       = bbg_data.loc[ticker, 'OPT_UNDL_PX']
    contract_size = bbg_data.loc[ticker, 'CONTRACT_SIZE']
    quantity      = abs(row['quantity'])
    fx_rate       = 0.89
    deriv_notional += delta * quantity * contract_size * undl_px * fx_rate

risk_df['gross_exposure'] = risk_df.apply(
    lambda r: deriv_notional if r['asset_class'] == 'Derivative'
    else (0.0 if r['asset_class'] == 'Cash'
    else r['abs_exposure']),
    axis=1
)

gross_leverage = risk_df['gross_exposure'].sum() / NAV

# ----------------------------------------------------------------
# Commitment method
# ----------------------------------------------------------------
long_eq  = risk_df[(risk_df['asset_class'] == 'Equity') &
                   (risk_df['market_value_eur'] > 0)]['market_value_eur'].sum()
short_eq = risk_df[(risk_df['asset_class'] == 'Equity') &
                   (risk_df['market_value_eur'] < 0)]['market_value_eur'].sum()
net_eq   = abs(long_eq + short_eq)
bonds    = risk_df[risk_df['asset_class'].isin(['Bond','Loan','CLO'])]['market_value_eur'].abs().sum()
fx       = risk_df[risk_df['asset_class'] == 'FX']['market_value_eur'].abs().sum()

commitment_exposure = net_eq + bonds + fx + deriv_notional
commitment_leverage = commitment_exposure / NAV

# ----------------------------------------------------------------
# Summary table
# ----------------------------------------------------------------
all_classes = sorted(risk_df['asset_class'].unique())

leverage_summary = pd.DataFrame({
    'Gross (EUR)'        : [risk_df[risk_df['asset_class']==ac]['gross_exposure'].sum()
                            for ac in all_classes],
    'Gross (x NAV)'      : [risk_df[risk_df['asset_class']==ac]['gross_exposure'].sum()/NAV
                            for ac in all_classes],
    'Commitment (EUR)'   : [risk_df[risk_df['asset_class']==ac]['market_value_eur'].abs().sum()
                            if ac not in ['Cash', 'Derivative'] else
                            (deriv_notional if ac == 'Derivative' else 0)
                            for ac in all_classes],
    'Commitment (x NAV)' : [risk_df[risk_df['asset_class']==ac]['market_value_eur'].abs().sum()/NAV
                            if ac not in ['Cash', 'Derivative'] else
                            (deriv_notional/NAV if ac == 'Derivative' else 0)
                            for ac in all_classes],
}, index=all_classes)

leverage_summary['Gross (EUR)']        = leverage_summary['Gross (EUR)'].map('{:,.0f}'.format)
leverage_summary['Gross (x NAV)']      = leverage_summary['Gross (x NAV)'].map('{:.2f}x'.format)
leverage_summary['Commitment (EUR)']   = leverage_summary['Commitment (EUR)'].map('{:,.0f}'.format)
leverage_summary['Commitment (x NAV)'] = leverage_summary['Commitment (x NAV)'].map('{:.2f}x'.format)

print(f"{'Asset Class':<15} {'Gross (EUR)':>15} {'Gross':>8} {'Commit (EUR)':>15} {'Commit':>8}")
print('-' * 65)
for ac in all_classes:
    row = leverage_summary.loc[ac]
    print(f"{ac:<15} {row['Gross (EUR)']:>15} {row['Gross (x NAV)']:>8} "
          f"{row['Commitment (EUR)']:>15} {row['Commitment (x NAV)']:>8}")
print('-' * 65)
print(f"{'Total':<15} {risk_df['gross_exposure'].sum():>15,.0f} {gross_leverage:>7.2f}x "
      f"{commitment_exposure:>15,.0f} {commitment_leverage:>7.2f}x")

GROSS_LIMIT = 3.0
status      = 'OK' if gross_leverage <= GROSS_LIMIT else 'BREACH'
print(f"\nGross leverage limit : {GROSS_LIMIT:.0f}x")
print(f"Current gross        : {gross_leverage:.2f}x")
print(f"Status               : {status}")

In [ ]:
# AIFMD II granular leverage breakdown
granular = risk_df.groupby(['asset_class', 'sub_asset_class']).agg(
    gross_eur=('gross_exposure', 'sum'),
    n_positions=('isin', 'count')
).reset_index()
granular['gross_x_nav'] = granular['gross_eur'] / NAV
granular['source']      = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[0], axis=1)
granular['listed_otc']  = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[1], axis=1)

granular = granular[granular['gross_eur'] > 0].sort_values('gross_eur', ascending=False)

# listed vs OTC summary
total_gross = granular['gross_eur'].sum()
summary_lot = granular.groupby('listed_otc')['gross_eur'].sum().reset_index()
summary_lot['x_nav']        = summary_lot['gross_eur'] / NAV
summary_lot['pct_leverage'] = summary_lot['gross_eur'] / total_gross * 100
summary_lot['gross_eur']    = summary_lot['gross_eur'].map('{:,.0f}'.format)
summary_lot['x_nav']        = summary_lot['x_nav'].map('{:.2f}x'.format)
summary_lot['pct_leverage'] = summary_lot['pct_leverage'].map('{:.1f}%'.format)
summary_lot.index.name      = None
summary_lot.columns         = ['Category', 'Gross (EUR)', 'x NAV', '% Leverage']
summary_lot.set_index('Category', inplace=True)

header = f"{'':12} {'Gross (EUR)':>15} {'x NAV':>8} {'% Leverage':>12}"
print(header)
print('-' * len(header))
for idx, row in summary_lot.iterrows():
    print(f"{idx:<12} {row['Gross (EUR)']:>15} {row['x NAV']:>8} {row['% Leverage']:>12}")
print('-' * len(header))
print()

summary_src = granular.groupby('source')['gross_eur'].sum().reset_index()
summary_src['x_nav']        = summary_src['gross_eur'] / NAV
summary_src['pct_leverage'] = summary_src['gross_eur'] / total_gross * 100
summary_src['gross_eur']    = summary_src['gross_eur'].map('{:,.0f}'.format)
summary_src['x_nav']        = summary_src['x_nav'].map('{:.2f}x'.format)
summary_src['pct_leverage'] = summary_src['pct_leverage'].map('{:.1f}%'.format)
summary_src.set_index('source', inplace=True)
summary_src.index.name      = None

header = f"{'':20} {'Gross (EUR)':>15} {'x NAV':>8} {'% Leverage':>12}"
print(header)
print('-' * len(header))
for idx, row in summary_src.iterrows():
    print(f"{idx:<20} {row['gross_eur']:>15} {row['x_nav']:>8} {row['pct_leverage']:>12}")
print('-' * len(header))
print()

# granular table
granular['pct_leverage'] = (granular['gross_eur'] / total_gross * 100).map('{:.1f}%'.format)
granular['gross_eur']    = granular['gross_eur'].map('{:,.0f}'.format)
granular['gross_x_nav']  = granular['gross_x_nav'].map('{:.2f}x'.format)
granular.set_index(['source', 'asset_class', 'sub_asset_class'], inplace=True)
granular

---

## 5. Stress Testing (Annex VI)

AIFMD Annex VI requires AIFMs to conduct regular stress tests covering market,
liquidity, and counterparty risk. Scenarios must be documented in the RMP,
reviewed at least annually, and results reported to the CSSF via Annex IV.

Unlike UCITS, no specific scenarios are prescribed. The scenarios below reflect
market risk stresses appropriate for a long/short equity fund with bond, FX,
and derivative exposure.

$$\Delta P_i = \text{sensitivity}_i \times \text{shock}_i \times MV_i$$

Scenarios covered:
- **Equity crash**: equity markets down 30%
- **Rate shock**: parallel shift up 200bps
- **Credit widening**: spreads widen 150bps
- **FX stress**: USD and GBP depreciate 15% vs EUR
- **Combined**: simultaneous equity, rate, credit and FX shock
- **Historical**: 2008 financial crisis, 2020 COVID crash, 2022 rate shock

> **Methodology note**: in this project, stress P&L uses first-order sensitivities (delta for
> equities, modified duration for rates and credit, direct revaluation for FX).
> In production these figures come from a third-party risk system or modeled to higher orders.

In [ ]:
from src.risk_utils import HISTORICAL_SCENARIOS

rows = []
for key, p in HISTORICAL_SCENARIOS.items():
    rows.append({
        'Scenario'    : p['name'],
        'Equity'      : f"{p['delta_equity']*100:.0f}%",
        'Rates (bps)' : f"{p['delta_y']*10000:.0f}",
        'Credit (bps)': f"+{p['delta_spread']*10000:.0f}",
        'USD'         : f"{p['fx_shocks'].get('USD', 0)*100:+.0f}%",
        'GBP'         : f"{p['fx_shocks'].get('GBP', 0)*100:+.0f}%",
    })

pd.DataFrame(rows).set_index('Scenario')

In [ ]:
# Individual stress scenarios
eq   = stress_equity(risk_df, delta_equity=-0.30)
rt   = stress_rates(risk_df, delta_y=0.02)
cr   = stress_credit(risk_df, delta_spread=0.015)
fx   = stress_fx(risk_df, fx_shocks={'USD': -0.15, 'GBP': -0.15})
cb   = stress_combined(risk_df)

# Historical scenarios
hist = {s: stress_historical(risk_df, s) for s in HISTORICAL_SCENARIOS}

# Summary table
rows = [
    {'Scenario': 'Equity Crash -30%',      'P&L (EUR)': eq['stressed_pnl_eur'],  '% NAV': eq['stressed_pnl_eur']/NAV*100},
    {'Scenario': 'Rate Shock +200bps',     'P&L (EUR)': rt['stressed_pnl_eur'],  '% NAV': rt['stressed_pnl_eur']/NAV*100},
    {'Scenario': 'Credit Widening +150bps','P&L (EUR)': cr['stressed_pnl_eur'],  '% NAV': cr['stressed_pnl_eur']/NAV*100},
    {'Scenario': 'FX Stress USD/GBP -15%', 'P&L (EUR)': fx['stressed_pnl_eur'],  '% NAV': fx['stressed_pnl_eur']/NAV*100},
    {'Scenario': 'Combined',               'P&L (EUR)': cb['stressed_pnl_eur'],  '% NAV': cb['stressed_pnl_eur']/NAV*100},
] + [
    {'Scenario': v['scenario'], 'P&L (EUR)': v['stressed_pnl_eur'], '% NAV': v['stressed_pnl_eur']/NAV*100}
    for v in hist.values()
]

summary_raw = pd.DataFrame(rows).set_index('Scenario')
worst_idx   = summary_raw['% NAV'].idxmin()

summary_raw['P&L (EUR)'] = summary_raw['P&L (EUR)'].map('{:,.0f}'.format)
summary_raw['% NAV']     = summary_raw['% NAV'].map('{:.2f}%'.format)

summary_raw.style.apply(lambda x: [
    'background-color: #7f1d1d; color: white' if i == worst_idx else ''
    for i in x.index], axis=0)

---
## 6. Liquidity Profile

ESMA guidelines (ESMA34-39-897) require AIFMs to categorise portfolio assets
by time to liquidation under normal market conditions. Results are reported
to the CSSF via Annex IV and used to assess redemption coverage.

Liquidity buckets (ESMA standard):

| Bucket | Time to liquidate |
|--------|------------------|
| 1      | 1 day            |
| 2      | 2-7 days         |
| 3      | 8-30 days        |
| 4      | 31-90 days       |
| 5      | 91-365 days      |
| 6      | > 1 year         |

ESMA buckets are roughly: day, week, month, quarter, year, or above.

Days to liquidate per position:

$$\text{days}_i = \frac{|MV_i|}{ADV_i \times \text{pct\_adv}}$$

* **ADV** (Average Daily Volume): average contracts/shares traded per day over
a 20-day window, sourced from Bloomberg. 
* **pct_adv = 25%** is an internal
RMP parameter representing the maximum fraction of ADV tradeable per day
without significant market impact. 
* Direct properties and instruments with
zero ADV are classified as illiquid (> 1 year).
* Cash and money market instruments are classified as 1 day by definition,
as they require no liquidation process.

In [ ]:
# MRS-30: Liquidity profile
risk_df_liq = days_to_liquidate(risk_df, pct_adv=0.25)
risk_df_liq = liquidity_buckets(risk_df_liq)

bucket_order = ['1 day', '2-7 days', '8-30 days', '31-90 days', '91-365 days', '> 1 year']

bucket_summary = risk_df_liq.groupby('liquidity_bucket').agg(
    market_value_eur=('market_value_eur', 'sum'),
    abs_exposure=('market_value_eur', lambda x: x.abs().sum()),
    n_positions=('isin', 'count')
).reset_index()
bucket_summary['pct_nav_net'] = bucket_summary['market_value_eur'] / NAV * 100
bucket_summary['pct_nav_abs'] = bucket_summary['abs_exposure'] / NAV * 100

bucket_full = bucket_summary.set_index('liquidity_bucket').reindex(bucket_order).fillna(0).reset_index()

# table
print(f"{'Bucket':<15} {'Abs Exposure (EUR)':>20} {'% NAV':>8} {'# Pos':>6}")
print('-' * 55)
for _, row in bucket_full.iterrows():
    if row['abs_exposure'] > 0:
        print(f"{row['liquidity_bucket']:<15} {row['abs_exposure']:>20,.0f} "
              f"{row['pct_nav_abs']:>7.1f}% {row['n_positions']:>6.0f}")
    else:
        print(f"{row['liquidity_bucket']:<15} {'—':>20} {'—':>8} {'—':>6}")
print('-' * 55)
total_abs = risk_df_liq['market_value_eur'].abs().sum()
print(f"{'Total':<15} {total_abs:>20,.0f} {total_abs/NAV*100:>7.1f}%")



In [ ]:
# chart
fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(bucket_full['liquidity_bucket'],
              bucket_full['pct_nav_abs'],
              color=ACCENT, alpha=0.85, width=0.5)
ax.axhline(0, color=C['dim'], lw=0.8)
ax.set_ylabel('Absolute Exposure (% NAV)', fontsize=9)
section_title(ax, f'Liquidity Profile — Absolute Exposure by Bucket', fontsize=10)

for bar, val in zip(bars, bucket_full['pct_nav_abs']):
    if val > 2:
        ax.text(bar.get_x() + bar.get_width()/2, val/2,
                f'{val:.1f}%', ha='center', va='center',
                fontsize=8, color='white', fontweight='bold')
plt.tight_layout()
save_fig(fig, FUND_ID, "04. Liquidity buckets HF")
plt.show()

---
### 6.2 Redemption Stress — MRS-31

AIFMD Article 16 and ESMA/2020/1498 require AIFMs to assess whether the
fund can meet redemptions under normal and stressed conditions. Assets
liquidatable within the contractual notice period (buckets `'1 day'` and
`'2-7 days'`) are compared against the redemption amount. A shortfall
triggers a gate or side-pocket recommendation to the risk committee.

| Scenario | Redemption | Notice | Rationale |
| --- | --- | --- | --- |
| Normal | 10% NAV | 5 days | Routine investor exit |
| Large | 25% NAV | 5 days | Large single redemption |
| Stress | 50% NAV | 5 days | Co-ordinated stress exit |
| Largest investor | Fund-specific | 5 days | Concentration stress |

In [ ]:
# MRS-31: Redemption stress — AIFM Hedge Fund
_NOTICE = 5   # contractual notice period (days)
_SCENARIOS = [
    (0.10, 'Normal    (10% NAV)'),
    (0.25, 'Large     (25% NAV)'),
    (0.50, 'Stress    (50% NAV)'),
]

print(f'Fund: AIFM Hedge Fund  |  NAV: EUR {NAV:,.0f}  |  Notice: {_NOTICE} days')
print()
print(f"{'':22} {'Redemption (M)':>14} {'Liquid (M)':>12} {'Gap (M)':>12} {'Coverage':>10} Action")
print('\u2500' * 95)

_redstress = {}
for _pct, _label in _SCENARIOS:
    _r = redemption_stress(risk_df_liq, NAV, redemption_pct=_pct, notice_days=_NOTICE)
    _redstress[_pct] = _r
    _gap = f"+{_r['liquidity_gap_eur']/1e6:.1f}M" if _r['liquidity_gap_eur'] >= 0 else f"{_r['liquidity_gap_eur']/1e6:.1f}M"
    print(f"{_label:<22} {_r['redemption_amount_eur']/1e6:>13.1f}M {_r['liquid_assets_eur']/1e6:>11.1f}M "
          f"{_gap:>12} {_r['coverage_ratio']:>9.2f}x  {_r['recommendation']}")
print('\u2500' * 95)
print('Largest-investor stress: see Section 6.3')

### 6.3 Investor Concentration — MRS-32

**ESMA thresholds** (ESMA/2020/1498, Annex VI):
- Single investor **> 20% of NAV** → concentration risk flag
- Top 3 investors **> 50% of NAV** → high-concentration flag

A concentrated investor base amplifies redemption risk: one large exit
can force asset sales that affect all remaining investors. The largest-investor
scenario below connects MRS-31 and MRS-32 — it uses the investor register to
derive the fourth redemption stress scenario.

In [ ]:
# MRS-32: Investor concentration — AIFM Hedge Fund
_investors = pd.DataFrame([
    {'investor_id': 'HF001', 'investor_name': 'Nordic Pension Fund', 'investor_type': 'Pension Fund', 'aum_eur': NAV * 0.25}, 
    {'investor_id': 'HF002', 'investor_name': 'Swiss Insurance Co', 'investor_type': 'Insurance', 'aum_eur': NAV * 0.17}, 
    {'investor_id': 'HF003', 'investor_name': 'European Family Office A', 'investor_type': 'Family Office', 'aum_eur': NAV * 0.13}, 
    {'investor_id': 'HF004', 'investor_name': 'German Asset Manager', 'investor_type': 'Asset Manager', 'aum_eur': NAV * 0.06}, 
    {'investor_id': 'HF005', 'investor_name': 'Endowment Fund B', 'investor_type': 'Endowment', 'aum_eur': NAV * 0.05}, 
    {'investor_id': 'HF006', 'investor_name': 'Sovereign Wealth Fund C', 'investor_type': 'Sovereign Wealth', 'aum_eur': NAV * 0.05}, 
    {'investor_id': 'HF007', 'investor_name': 'Private Bank D', 'investor_type': 'Private Bank', 'aum_eur': NAV * 0.04}, 
    {'investor_id': 'HF008', 'investor_name': 'Insurance Co E', 'investor_type': 'Insurance', 'aum_eur': NAV * 0.04}, 
    {'investor_id': 'HF009', 'investor_name': 'Fund of Funds F', 'investor_type': 'FoF', 'aum_eur': NAV * 0.04}, 
    {'investor_id': 'HF010', 'investor_name': 'Family Office G', 'investor_type': 'Family Office', 'aum_eur': NAV * 0.03}, 
    {'investor_id': 'HF011', 'investor_name': 'HNWI H', 'investor_type': 'HNWI', 'aum_eur': NAV * 0.03}, 
    {'investor_id': 'HF012', 'investor_name': 'Pension Fund I', 'investor_type': 'Pension Fund', 'aum_eur': NAV * 0.03}, 
    {'investor_id': 'HF013', 'investor_name': 'Family Office J', 'investor_type': 'Family Office', 'aum_eur': NAV * 0.03}, 
    {'investor_id': 'HF014', 'investor_name': 'HNWI K', 'investor_type': 'HNWI', 'aum_eur': NAV * 0.03}, 
    {'investor_id': 'HF015', 'investor_name': 'Multi-Family Office L', 'investor_type': 'Family Office', 'aum_eur': NAV * 0.02}, 
])

_conc = investor_concentration(_investors, NAV, threshold=0.20)
_top  = _conc['by_investor']
_type = _investors.set_index('investor_id')['investor_type']

print(f'Investor Concentration — AIFM Hedge Fund  |  NAV: EUR {NAV:,.0f}')
print('ESMA threshold: 20% single / 50% top-3\n')
print(f"{'':4} {'Investor':<30} {'Type':<18} {'AUM (EUR)':>14} {'% NAV':>8}")
print('\u2500' * 80)
for _rank, (_, _row) in enumerate(_top.iterrows(), 1):
    _t = _type.get(_row['investor_id'], '')
    print(f"{_rank:<4} {_row['investor_name']:<30} {_t:<18} {_row['aum_eur']:>14,.0f} {_row['pct_nav']*100:>7.1f}%")
print('\u2500' * 80)

_flag_s = '\u26a0 ESMA flag'       if _conc['concentration_flag'] else '\u2713 OK'
_flag_3 = '\u26a0 High conc.'      if _conc['high_concentration'] else '\u2713 OK'
print(f"\nLargest investor : {_conc['largest_investor_pct']*100:.1f}% NAV  {_flag_s}")
print(f"Top 3 investors  : {_conc['top3_pct']*100:.1f}% NAV  {_flag_3}")

# Largest-investor redemption stress (4th scenario)
_r4   = redemption_stress(risk_df_liq, NAV, redemption_pct=_conc['largest_investor_pct'], notice_days=5)
_gap4 = f"+{_r4['liquidity_gap_eur']/1e6:.1f}M" if _r4['liquidity_gap_eur'] >= 0 else f"{_r4['liquidity_gap_eur']/1e6:.1f}M"
print(f"\nLargest-investor stress ({_conc['largest_investor_pct']*100:.1f}% NAV, 5-day notice):")
print(f"  Redemption : EUR {_r4['redemption_amount_eur']:,.0f}")
print(f"  Liquid     : EUR {_r4['liquid_assets_eur']:,.0f}")
print(f"  Gap        : {_gap4}  |  Coverage: {_r4['coverage_ratio']:.2f}x")
print(f"  Action     : {_r4['recommendation']}")

print('\nMonitoring recommendation:')
if _conc['high_concentration']:
    print('  \u2014 Enhanced monitoring: top-3 investors represent significant co-ordinated exit risk')
    print('  \u2014 Maintain liquidity buffer >= largest investor AUM')
if _conc['concentration_flag']:
    print(f'  \u2014 Gate-trigger review: largest investor at {_conc["largest_investor_pct"]*100:.1f}% NAV')
if not _conc['concentration_flag'] and not _conc['high_concentration']:
    print('  \u2014 No immediate action. Continue quarterly investor concentration monitoring.')

### 6.4 Counterparty Stress

AIFMD Annex VI requires AIFMs to stress-test **counterparty default** scenarios. For a long/short equity and credit strategy the key counterparties are the prime broker(s) and OTC derivatives (ISDA) counterparties. The relevant exposure is the **net uncollateralised** position after applying initial margin and variation margin held at the counterparty.

> Collateral coverage is simulated. A production system would derive these figures from the daily collateral reconciliation.


In [ ]:
# MRS-35: Counterparty stress — AIFM Hedge Fund
# Simulated prime brokerage and ISDA OTC derivatives counterparty register
_cp_hf = pd.DataFrame([
    {'counterparty': 'Goldman Sachs',  'type': 'Prime Broker',    'exposure_pct': 0.12, 'collateral_cover': 0.80},
    {'counterparty': 'Morgan Stanley', 'type': 'Prime Broker',    'exposure_pct': 0.08, 'collateral_cover': 0.80},
    {'counterparty': 'Deutsche Bank',  'type': 'ISDA/OTC Deriv.', 'exposure_pct': 0.05, 'collateral_cover': 0.70},
    {'counterparty': 'BNP Paribas',    'type': 'ISDA/OTC Deriv.', 'exposure_pct': 0.04, 'collateral_cover': 0.70},
])
_cp_hf['exposure_eur']     = _cp_hf['exposure_pct'] * NAV
_cp_hf['collateral_eur']   = _cp_hf['exposure_eur'] * _cp_hf['collateral_cover']
_cp_hf['net_exposure_eur'] = _cp_hf['exposure_eur'] * (1 - _cp_hf['collateral_cover'])
_cp_hf['loss_pct_nav']     = _cp_hf['net_exposure_eur'] / NAV

_worst_cp    = _cp_hf.loc[_cp_hf['net_exposure_eur'].idxmax()]
_cp_loss_eur = _worst_cp['net_exposure_eur']
_cp_loss_pct = _worst_cp['loss_pct_nav']

print(f"Counterparty Stress — AIFM Hedge Fund  |  NAV: EUR {NAV:,.0f}")
print(f"Simulated prime brokerage and OTC derivatives counterparty register\n")
print(f"{'Counterparty':<18} {'Type':<16} {'Exposure':>12} {'Collateral':>12} {'Net Exp.':>12} {'% NAV':>8}")
print('─' * 82)
for _, r in _cp_hf.iterrows():
    print(f"{r['counterparty']:<18} {r['type']:<16} {r['exposure_eur']:>11,.0f} "
          f"{r['collateral_eur']:>11,.0f} {r['net_exposure_eur']:>11,.0f} {r['loss_pct_nav']*100:>7.1f}%")
print('─' * 82)
print(f"\nWorst-case: {_worst_cp['counterparty']} defaults")
print(f"  Net loss (post-collateral): EUR {_cp_loss_eur:,.0f}  ({_cp_loss_pct*100:.1f}% of NAV)")
print(f"  AIFMD limit: no single counterparty net exposure > 5% NAV (UCITS/AIFM guideline)")
_flag_cp = "⚠ BREACH" if _cp_loss_pct > 0.05 else "✓ Within limit"
print(f"  Status: {_flag_cp}")


### 6.5 Combined Stress (Market + Liquidity) — MRS-35

ESMA/2020/1498 Annex VI requires a **combined stress** scenario: what happens if the market falls and investors simultaneously redeem? For this fund the combined shock is **equity −20%** (market stress) **and** **25% NAV redemption** (liquidity stress) applied simultaneously.

Under combined stress, liquid assets also decline in value — equity held as collateral or as liquidatable positions shrinks by 20%. The redemption demand is computed on the **pre-stress NAV** (investor expectation at point of submission).


In [ ]:
# MRS-35: Combined stress — equity -20% AND 25% redemption simultaneously
_comb_eq      = stress_equity(risk_df, delta_equity=-0.20)
_comb_mkt_eur = _comb_eq['stressed_pnl_eur']
_comb_nav_st  = NAV + _comb_mkt_eur

# Liquid assets shrink by 20% under equity stress
_base_red           = _redstress[0.25]
_comb_liquid_st     = _base_red['liquid_assets_eur'] * (1 - 0.20)
_comb_redeem_eur    = NAV * 0.25
_comb_gap_st        = _comb_liquid_st - _comb_redeem_eur
_comb_cov_st        = _comb_liquid_st / _comb_redeem_eur if _comb_redeem_eur > 0 else float('inf')
_comb_action        = 'Can meet redemption' if _comb_gap_st >= 0 else 'Gate / partial suspension required'

print(f"Combined Stress — AIFM Hedge Fund  |  Equity −20% + 25% Redemption")
print(f"Baseline NAV: EUR {NAV/1e6:,.1f}M\n")
print(f"  Market shock (equity −20%):")
print(f"    Portfolio P&L  : EUR {_comb_mkt_eur/1e6:,.1f}M  ({_comb_mkt_eur/NAV*100:.1f}% NAV)")
print(f"    Stressed NAV   : EUR {_comb_nav_st/1e6:,.1f}M")
print()
print(f"  Liquidity impact (25% redemption, liquid assets stressed −20%):")
print(f"    Redemption     : EUR {_comb_redeem_eur/1e6:,.1f}M  (25% pre-stress NAV)")
print(f"    Liquid assets  : EUR {_comb_liquid_st/1e6:,.1f}M  (post equity shock)")
print(f"    Liquidity gap  : EUR {_comb_gap_st/1e6:,.1f}M  |  Coverage: {_comb_cov_st:.2f}x")
print(f"    Action         : {_comb_action}")
print()
_total_stress = _comb_mkt_eur - max(0.0, -_comb_gap_st)
_total_pct    = _total_stress / NAV * 100
print(f"  Total combined impact on NAV: EUR {_total_stress/1e6:,.1f}M  ({_total_pct:.1f}% of NAV)")
print(f"  Regulatory note: ESMA/2020/1498 §48 — combined stress is a mandatory Annex VI scenario")


## 7. P&L Attribution by Risk Factor

Under AIFMD Article 15, the risk function must be able to explain where
returns and losses come from — not just report the total NAV move. The CSSF
expects the risk manager to answer "what drove the loss on day X" by factor.

This section uses **sensitivity-based attribution**. Regression-based
approaches give average historical loadings and cannot reflect current
position changes. Sensitivity-based attribution uses actual positions and
actual market moves each day, consistent with how VaR is computed.

> This is a risk governance output, not a direct Annex IV or Annex VI field.
> It feeds the Board risk report and supports the AIFMD Article 15 evidence pack.

### Attribution framework

#### A. Total

$\text{Total P\&L} = \text{Equity P\&L} + \text{Rates P\&L} + \text{FX P\&L} + \text{Residual}$

---

#### B. Components

$\text{Equity P\&L} = \sum_i \beta_i \times r_{market} \times MV_i$

$\text{Rates P\&L} = \sum_i -D_i \times \Delta y \times MV_i$

$\text{FX P\&L} = \sum_i \text{notional}_{foreign,i} \times r_{FX,i}$

where $\beta_i$ is the 1-year rolling beta to the benchmark, $D_i$ is
modified duration, and $r_{FX,i}$ is the daily FX return vs EUR.

---

#### C. Residual

$\text{Residual} = \text{P\&L}_{actual} - (\text{Equity} + \text{Rates} + \text{FX})$

A large or persistent residual signals model limitations, missing factors
(credit spread, volatility, carry), or data issues. It is shown, not suppressed.

**% explained** $= |\text{explained}| / |\text{actual}|$. Target $\geq 80\%$.

In [ ]:
import sqlalchemy as sa
from src.risk_utils import compute_pnl_attribution
from src.database import query_nav_history

# Actual daily P&L
nav_history = query_nav_history(ENGINE, FUND_ID)
nav_history['date'] = pd.to_datetime(nav_history['date'])
nav_history = nav_history.set_index('date').sort_index()
pnl_actual  = nav_history['pnl_eur'].dropna()

START_DATE         = pnl_actual.index.min().strftime('%Y-%m-%d')
VALUATION_DATE_STR = VALUATION_DATE

# Benchmark — SPY for USD-heavy long/short portfolio
spy_bm = BBG.bdh('SPY US Equity', 'PX_LAST', START_DATE, VALUATION_DATE_STR)
spy_bm['r_market'] = spy_bm['PX_LAST'].pct_change()

# Rate shift — simulated parallel USD curve move
np.random.seed(42)
rate_series = pd.Series(
    np.random.normal(0, 0.0005, len(spy_bm)),
    index=spy_bm.index, name='dy'
)

# FX
usd = BBG.bdh('EURUSD Curncy', 'PX_LAST', START_DATE, VALUATION_DATE_STR)
usd['r_fx_USD'] = usd['PX_LAST'].pct_change()

# Market moves
market_moves = pd.DataFrame(index=spy_bm.index)
market_moves['r_market'] = spy_bm['r_market']
market_moves['dy']       = rate_series
market_moves['r_fx_USD'] = usd['r_fx_USD']
market_moves = market_moves.dropna()

# Daily positions history
with ENGINE.connect() as conn:
    positions_history = pd.read_sql(sa.text("""
        SELECT p.date, p.isin, p.asset_class, p.currency,
               p.market_value_eur, pe.beta, pe.dur_adj_mid
        FROM positions p
        LEFT JOIN positions_enriched pe
            ON p.isin = pe.isin AND p.fund_id = pe.fund_id
        WHERE p.fund_id = :fund_id
        ORDER BY p.date
    """), conn, params={'fund_id': FUND_ID})

positions_history['date'] = pd.to_datetime(positions_history['date'])

common_dates     = market_moves.index.intersection(pnl_actual.index)
market_moves_aln = market_moves.loc[common_dates]
pnl_aligned      = pnl_actual.loc[common_dates]
pos_history_aln  = positions_history[positions_history['date'].isin(common_dates)]

attr = compute_pnl_attribution(pos_history_aln, market_moves_aln, pnl_aligned)

# Cumulative attribution chart
fig, ax = plt.subplots(figsize=(11, 5))
cum = attr[['pnl_equity', 'pnl_rates', 'pnl_fx', 'pnl_residual']].cumsum() / 1e6
ax.plot(cum.index, cum['pnl_equity'],   color=ACCENT,    linewidth=1.5, label='Equity')
ax.plot(cum.index, cum['pnl_rates'],    color=ACCENT2,   linewidth=1.5, label='Rates')
ax.plot(cum.index, cum['pnl_fx'],       color=ACCENT3,   linewidth=1.5, label='FX')
ax.plot(cum.index, cum['pnl_residual'], color=C['red'], linewidth=1.0,
        linestyle='--', label='Residual')
ax.axhline(0, color='white', linewidth=0.5, linestyle='--')
ax.set_ylabel('Cumulative P&L (EUR MM)')
section_title(ax, f'Cumulative P&L Attribution by Risk Factor — AIFM Hedge Fund', fontsize=18)
ax.legend(fontsize=9)
plt.tight_layout()
save_fig(fig, FUND_ID, "05. PnL attribution HF")
plt.show()


# Model quality
RESIDUAL_THRESHOLD_PCT = 0.20
resid_vol = attr['pnl_residual'].std()
flagged = attr[
    (attr['pct_explained'].notna()) &
    (
        (1 - attr['pct_explained'] > RESIDUAL_THRESHOLD_PCT) |
        (attr['pnl_residual'].abs() > 2.0 * resid_vol)
    )
].copy()

print(f"{'Attribution period':<35} {attr.index.min().date()} to {attr.index.max().date()}")
print(f"{'Days attributed':<35} {len(attr)}")
print(f"{'Correlation (actual vs expl.)':<35} {attr['pnl_actual'].corr(attr['pnl_explained']):.3f}")
print(f"{'Median % explained':<35} {attr['pct_explained'].median():.1%}")
print(f"{'Days >= 80% explained':<35} {(attr['pct_explained'] >= 0.80).sum()} ({(attr['pct_explained'] >= 0.80).mean():.1%})")
print(f"{'Residual vol (EUR)':<35} {attr['pnl_residual'].std():,.0f}")
print(f"{'Residual / total vol':<35} {attr['pnl_residual'].std() / attr['pnl_actual'].std():.1%}")
print(f"{'Flagged days':<35} {len(flagged)} ({len(flagged)/len(attr):.1%})")
print()
print("Note: residual = idiosyncratic return not explained by market beta.")
print("For a long/short equity fund the residual represents alpha —")
print("stock-specific return from security selection independent of")
print("market direction. Persistent positive residual = successful stock picking.")




**Methodology and limitations**

Sensitivity-based attribution using actual daily positions and market moves.
Regression-based attribution is not used — it gives average historical loadings
and cannot reflect current position composition. 

Benchmark: SPY (S&P 500) — appropriate for this USD-heavy long/short equity portfolio.

**Limitations:**
* Single-factor equity model — no sector, style, or stock-level decomposition
* Rate attribution uses a simulated parallel shift — no key-rate DV01
* FX covers EUR/USD only
* Position composition is static in this simulation

**Regulatory context:** this section is an internal governance output consumed
by the Board and risk committee. It supports the AIFMD Article 15 evidence
pack and CSSF expectations for risk management reporting. It is not a direct
Annex IV or Annex VI deliverable.

## 8. Pre-Trade Compliance Check — MRS-61

Pre-trade compliance is a front-line risk management control. Before the portfolio manager \
executes, the risk function must confirm the proposed trade will not breach fund limits. \
Under AIFMD Article 15, the AIFM must have documented risk management procedures — \
pre-trade compliance is a key operational part of that framework.

`pre_trade_check` (in `risk_utils.py`) works in three steps:
1. Loads the current enriched portfolio via `get_risk_ready_df`
2. Constructs a pro-forma position set after applying the proposed trade (`_ptc_apply_trade`)
3. Runs AIFM Hedge Fund checks: gross leverage, commitment leverage, single-issuer \
concentration, sector concentration, counterparty exposure (OTC), short-selling \
reportability (EU 236/2012), and liquidity impact

**Regulatory basis:** EU 231/2013 Articles 6–8 (gross and commitment leverage methods), \
AIFMD Article 15 (risk management), EU Regulation 236/2012 (short selling).

In [ ]:
from src.risk_utils import pre_trade_check


def _show_ptc(result: dict) -> None:
    status = "✓  PASSED" if result['passed'] else "✗  FAILED"
    t = result['proposed_trade']
    notional = abs(t['quantity'] * t['price_eur'])
    print(f"\\n{'─'*62}")
    print(f"  Pre-trade check  │  {result['fund_id']}")
    print(f"  Trade            │  {t['direction'].upper()} {t['quantity']:,} × {t['isin']}")
    print(f"                   │  @ EUR {t['price_eur']:,.2f}  (notional EUR {notional:,.0f})")
    print(f"  Result           │  {status}")
    print(f"{'─'*62}")
    print("  Post-trade metrics:")
    for k, v in result['post_trade_metrics'].items():
        if isinstance(v, float):
            print(f"    {k:<38}: {v:.4f}")
        else:
            print(f"    {k:<38}: {v}")
    if result['breaches']:
        print(f"\\n  Breaches detected ({len(result['breaches'])}):")
        for b in result['breaches']:
            print(f"\\n    ⚠  {b['check']}")
            print(f"       Limit   : {b['limit']} {b['unit']}")
            print(f"       Actual  : {b['actual']} {b['unit']}")
            print(f"       Detail  : {b['message']}")
    else:
        print("\\n  No limit breaches — trade approved.")
    print()

### 8.1  Scenario 1 — Clean buy (passes all checks)

New investment-grade bond, issuer not currently in the portfolio. \
EUR 3M notional on a fund NAV of approximately EUR 94.5M (≈ 3.2% of NAV). \
Post-trade gross leverage stays well below 300%; new issuer is below the 25% RMP concentration \
limit; eligible instrument. **Expected result: PASS.**

In [ ]:
# Deutsche Bank IG bond — new name, small size
trade_pass = {
    'isin'           : 'XS2500000001',
    'direction'      : 'buy',
    'quantity'       : 3_000_000,
    'price_eur'      : 1.00,
    'asset_class'    : 'Bond',
    'sub_asset_class': 'IG Corporate',
    'dur_adj_mid'    : 4.2,
    'currency'       : 'EUR',
    'adv_eur'        : 5_000_000,
    'issuer'         : 'Deutsche Bank AG',
}

result_pass = pre_trade_check(ENGINE, FUND_ID, trade_pass, VALUATION_DATE)
_show_ptc(result_pass)

### 8.2  Scenario 2 — Gross and commitment leverage breach

A large new long equity position: 750,000 shares of Goldman Sachs at EUR 200/share \
= EUR 150M notional. The fund's current gross exposure is approximately EUR 152M on a \
NAV of EUR 94.5M (≈ 1.6×). Adding EUR 150M pushes gross exposure to ≈ EUR 302M = **3.20× NAV**, \
exceeding the 300% RMP limit. Commitment leverage also breaches. \
The single new issuer at EUR 150M would additionally represent ≈ 158% NAV, \
far above the 25% single-issuer concentration limit.

> This scenario deliberately stacks multiple breaches: it shows that `pre_trade_check` \
catches all of them simultaneously and presents each with its limit and actual value.

In [ ]:
# Goldman Sachs — oversized long, breaches gross leverage and issuer concentration
trade_leverage = {
    'isin'           : 'US38141G1040',
    'direction'      : 'buy',
    'quantity'       : 750_000,
    'price_eur'      : 200.00,
    'asset_class'    : 'Equity',
    'sub_asset_class': 'Large Cap',
    'beta'           : 1.40,
    'currency'       : 'USD',
    'adv_eur'        : 150_000_000,
    'issuer'         : 'Goldman Sachs Group',
}

result_lev = pre_trade_check(ENGINE, FUND_ID, trade_leverage, VALUATION_DATE)
_show_ptc(result_lev)

### 8.3  Scenario 3 — Single-issuer concentration breach only

Alphabet Inc is not currently in the portfolio. Buying 150,000 shares at EUR 170/share \
= EUR 25.5M notional. On NAV of EUR 94.5M, this is **27.0%**, exceeding the 25% RMP \
single-issuer limit. The size is deliberately chosen to breach concentration while \
staying well within the 300% gross leverage limit (post-trade gross ≈ 1.88× NAV).

This is the typical operational question: the portfolio manager wants to open a meaningful \
new position. The risk function must confirm the proposed allocation is within limits before \
the order reaches the market. A minor reduction in size (below EUR 23.6M = 25% NAV) \
would resolve the breach.

In [ ]:
# Alphabet Inc — new position, sized to breach issuer concentration (27% NAV)
trade_conc = {
    'isin'           : 'US02079K3059',
    'direction'      : 'buy',
    'quantity'       : 150_000,           # 150 k × EUR 170 = EUR 25.5 M
    'price_eur'      : 170.00,
    'asset_class'    : 'Equity',
    'sub_asset_class': 'Large Cap',
    'beta'           : 1.15,
    'currency'       : 'USD',
    'adv_eur'        : 200_000_000,
    'issuer'         : 'Alphabet Inc',
}

result_conc = pre_trade_check(ENGINE, FUND_ID, trade_conc, VALUATION_DATE)
_show_ptc(result_conc)

----


## ESG Risk Indicators

ESG monitoring is required under CSSF Regulation 22-05 and SFDR (EU 2019/2088).
Portfolio-level ESG scores are computed as NAV-weighted averages. 

**SFDR classification**: Article 6 (no sustainability claim). ESG factors are
monitored but do not drive investment decisions. Article 8 would require documented
ESG screening; Article 9 would require sustainable investment as the primary objective.

Metrics monitored:
- Weighted average ESG score (composite, E, S, G)
- % of NAV in instruments with ESG score below 30*
- % of NAV with active controversy flag
- Weighted average carbon intensity (tCO2/EURm revenue)

> **Note**: ESG scores for liquid instruments are fetched from Bloomberg at
> enrichment time. Instruments without a Bloomberg ticker use fund administrator
> ESG data embedded in the position file.


> % of NAV in instruments with ESG score below internal threshold. 
> Threshold is defined in the Risk Management Policy. 
> ESG scores are not comparable across providers as each
> uses a different methodology and scale.
> (here using 30/100,Bloomberg scale 0-100 higher is better)

> **Scale note**: all ESG scores in this framework use a 0-100 scale where
> 100 is best, consistent with Bloomberg ESG methodology. Illiquid instrument
> scores are provided by the fund administrator on the same scale. In production,
> the ManCo should ensure all third-party ESG data is normalised to a consistent
> scale before aggregating portfolio-level metrics.

> **ESG look-through for derivatives**: derivatives have indirect ESG exposure
> through their underlying. The delta-adjusted notional is used as the ESG
> exposure weight rather than market value, which would understate the exposure.
>
> For options:
> $$ESG\_exposure_i = |\delta_i| \times Q_i \times \text{contract size} \times P_{underlying} \times FX$$
>
> For futures: full notional is used since delta = 1.
>
> For FX forwards: no ESG exposure assigned (no issuer).

In [ ]:
esg_df  = build_esg_df(risk_df, BBG, ENGINE, FUND_ID, VALUATION_DATE)
summary = esg_portfolio_summary(esg_df, NAV)

In [ ]:
def style_esg(row):
    styles = [''] * len(row)
    idx = esg_df.columns.tolist()
    if row.get('controversy_flag') == True:
        return ['background-color: #7f1d1d; color: white'] * len(row)
    elif pd.notna(row.get('esg_score')) and row.get('esg_score') < ESG_THRESHOLD:
        styles[idx.index('esg_score')] = 'background-color: #7f1d1d; color: white'
    return styles

esg_df.style.apply(style_esg, axis=1).format({
    'market_value_eur' : '{:,.0f}',
    'weight_pct'       : '{:.2f}%',
    'esg_score'        : '{:.0f}',
    'env_score'        : '{:.0f}',
    'soc_score'        : '{:.0f}',
    'gov_score'        : '{:.0f}',
    'carbon_intensity' : '{:.1f}',
    'esg_exposure_eur' : '{:,.0f}',
}, na_rep='—')

In [ ]:
print(f"ESG PORTFOLIO SUMMARY")
print('-' * 45)
print(f"{'Weighted avg ESG score':<30} {summary['wav_esg']:.1f}/100")
print(f"{'Weighted avg ENV score':<30} {summary['wav_env']:.1f}/100")
print(f"{'Weighted avg SOC score':<30} {summary['wav_soc']:.1f}/100")
print(f"{'Weighted avg GOV score':<30} {summary['wav_gov']:.1f}/100")
print(f"{'Weighted avg carbon intensity':<30} {summary['wav_carbon']:.1f} tCO2/EURm")
print(f"{'% exposure below ESG threshold':<30} {summary['pct_low_esg']:.1f}%")
print(f"{'% exposure with controversy':<30} {summary['pct_controversy']:.1f}%")
print()
if len(summary['controversies']) > 0:
    print("Controversy flags:")
    for _, row in summary['controversies'].iterrows():
        print(f"  {row['instrument_name']:<35} ESG: {row['esg_score']:.0f}")

In [ ]:
esg_scored = esg_df[esg_df['esg_score'].notna()].copy()
total_scored_mv = esg_scored['esg_exposure_eur'].sum()
ac_esg = esg_scored.groupby('asset_class').agg(
    wav_esg=('esg_score', lambda x: (x * esg_scored.loc[x.index, 'esg_exposure_eur']).sum() /
             esg_scored.loc[x.index, 'esg_exposure_eur'].sum()),
    exposure=('esg_exposure_eur', 'sum')
).reset_index()
ac_esg['pct_total'] = ac_esg['exposure'] / total_scored_mv * 100

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
sup_title(fig, 'ESG Profile by Asset Class', fontsize=18)

colors = [C['muted'], C['dim'], C['border'], C['border'], C['text'], C['text']]
left = 0
for i, (_, row) in enumerate(ac_esg.iterrows()):
    axes[0].barh(0, row['pct_total'], left=left,
                 color=colors[i % len(colors)], alpha=0.85, height=0.2)
    if row['pct_total'] > 3:
        axes[0].text(left + row['pct_total']/2, 0,
                     f"{row['asset_class']}\n{row['pct_total']:.1f}%",
                     ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    left += row['pct_total']

axes[0].set_xlim(0, 100)
axes[0].set_yticks([])
axes[0].set_xlabel('% of ESG-scored exposure', fontsize=9)
axes[0].spines[['top', 'right', 'left', 'bottom']].set_visible(False)
axes[0].tick_params(labelsize=9, length=0)

bars = axes[1].barh(ac_esg['asset_class'], ac_esg['wav_esg'],
                    color=[C['amber'] if v < 50 else C['blue2'] if v < 70 else C['green']
                           for v in ac_esg['wav_esg']],
                    height=0.4, alpha=0.85)
axes[1].axvline(ESG_THRESHOLD, color=ACCENT2, lw=1, linestyle='--',
                label=f'Low ESG threshold ({ESG_THRESHOLD})')
axes[1].axvline(70, color=ACCENT3, lw=1, linestyle='--', label='Good ESG threshold (70)')
axes[1].set_xlim(0, 100)
axes[1].set_xlabel('Weighted avg ESG score', fontsize=9)
axes[1].spines[['top', 'right', 'left', 'bottom']].set_visible(False)
axes[1].grid(True, axis='x', alpha=0.15, linestyle='--')
axes[1].tick_params(labelsize=9, length=0)
axes[1].legend(fontsize=7)
for bar, val in zip(bars, ac_esg['wav_esg']):
    axes[1].text(val + 1, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9)
plt.tight_layout()
save_fig(fig, FUND_ID, "06. ESG profile - HF")
plt.show()



In [ ]:
attr[['pnl_actual', 'pnl_explained', 'pnl_residual']].describe()

In [ ]:
total_mv = risk_df['market_value_eur'].abs().sum()
unattributed = risk_df[
    risk_df['asset_class'].isin(['Derivative', 'Cash'])
]['market_value_eur'].abs().sum()
print(f"Total MV:        EUR {total_mv:,.0f}")
print(f"Unattributed MV: EUR {unattributed:,.0f} ({unattributed/total_mv:.1%})")
print(f"\nBy asset class:")
print(risk_df.groupby('asset_class')['market_value_eur'].apply(
    lambda x: x.abs().sum()
).sort_values(ascending=False))

In [ ]:
print(f"Dates in positions table: {len(attr)}")
print(f"Unique ISINs over history:")
from src.database import query_positions
pos_all = query_positions(ENGINE, FUND_ID, VALUATION_DATE)
print(pos_all.groupby('date')['isin'].count().describe())

In [ ]:
print(attr[['pnl_actual', 'pnl_explained']].corr())
print(f"\nCorrelation: {attr['pnl_actual'].corr(attr['pnl_explained']):.3f}")

---

## 9. Annex IV Report — MRS-34

AIFMD Article 110 requires the AIFM to submit a quarterly transparency report
to the CSSF (Annex IV template). The report covers fund identity, strategy,
risk profile, leverage (gross and commitment method), and liquidity.
ESMA technical guidance v1.7 (July 2024) defines the mandatory field set.

AIFMD II (Directive 2024/927/EU) expanded the leverage disclosure to a granular
breakdown by source: financial borrowing, synthetic leverage through derivatives,
and repo/reverse repo — making it easier for the CSSF to identify funds building
leverage through derivatives rather than direct borrowing.

**Regulatory basis:** AIFMD Art. 110 / EU 231/2013 Annex IV / ESMA v1.7 (Jul 2024)


In [ ]:
from src.annex_iv import build_annex_iv, export_annex_iv_excel

ANNEX_IV_QUARTER = '2026-03-31'

annex_iv = build_annex_iv(ENGINE, FUND_ID, quarter=ANNEX_IV_QUARTER)
print(f"Annex IV — {FUND_ID}   Reporting period: Q1 2026 ({ANNEX_IV_QUARTER})")
print(f"NAV: EUR {annex_iv['_nav']/1e6:,.1f}M")

# Section 1 — Fund identification
print("\n=== Section 1 — Fund Identification ===")
display(annex_iv['identification'])


In [ ]:
from src.nb_utils import styled_table

# Section 2 — Exposures
styled_table(annex_iv['asset_class_breakdown'],
             title='2.1  Asset Class Breakdown')
styled_table(annex_iv['geography_breakdown'],
             title='2.2  Geographic Breakdown')
styled_table(annex_iv['currency_breakdown'],
             title='2.3  Currency Breakdown')
styled_table(annex_iv['top5_positions'],
             title='2.4  Top 5 Positions')


In [ ]:
# Section 3 — Risk measures (VaR, leverage summary, liquidity headline)
print("\n=== Section 3 — Risk Measures ===")
display(annex_iv['risk_measures'])

# Section 4 — Leverage detail (gross breakdown by source + commitment)
print("\n=== Section 4 — Leverage Detail ===")
display(annex_iv['leverage_detail'])

# Section 5 — Liquidity (ESMA buckets + redemption terms + investor concentration)
print("\n=== Section 5 — Liquidity Profile ===")
liq = annex_iv['liquidity_buckets']
cols = [c for c in ['bucket', 'nav_eur', 'nav_pct', 'cumulative_pct'] if c in liq.columns]
display(liq[cols])

print("\n=== Section 5 — Redemption Terms & Investor Concentration ===")
display(annex_iv['liquidity_terms'])


In [ ]:
# Export to Excel — CSSF submission format
path = export_annex_iv_excel(ENGINE, quarter=ANNEX_IV_QUARTER)
print(f"Annex IV workbook written: {path}")
